## basics

- Spectral Normalization for Generative Adversarial Networks
    - https://arxiv.org/abs/1802.05957
- pytorch 的两个接口
    - old：`torch.nn.utils.spectral_norm`
    - new：`torch.nn.utils.parametrizations.spectral_norm`
    - https://docs.pytorch.org/docs/2.12/generated/torch.nn.utils.spectral_norm.html#torch.nn.utils.spectral_norm
    - https://docs.pytorch.org/docs/2.12/generated/torch.nn.utils.parametrizations.spectral_norm.html#torch.nn.utils.parametrizations.spectral_norm

## 矩阵的谱范数

$$
\|A\|_2
=
\max_{x\neq 0}
\frac{\|Ax\|_2}{\|x\|_2}
=
\sqrt{\lambda_{\max}(A^\top A)}
=
\sigma_{\max}(A)
$$

谱归一化通过使用幂迭代法（torch api中使用的方法）计算权重矩阵的谱范数来重新调整权重张量，从而稳定生成对抗网络（GANs）中判别器（critics）的训练。

- A的谱范数 = A的最大奇异值 = A^T·A的最大特征值的平方根
- 谱范数（也称为诱导2-范数）是矩阵的最大奇异值。直观上，它表示矩阵将向量“拉伸”的最大倍数。The spectral norm (also known as the induced 2-norm) is the maximum singular value of a matrix. Intuitively, you can think of it as the maximum "scale", by which the matrix can "stretch" a vector.
- 最大奇异值等于 A^\top A 的最大特征值的平方根；如果矩阵是对称/厄米的，则等于其最大的特征值。The maximum singular value is the square root of the maximum eigenvalue of A^\top A (and equals the maximum eigenvalue if the matrix is symmetric/Hermitian).

- 两种计算方法
    - svd 分解
    - 幂迭代法 (Power Iteration Method)

In [1]:
import numpy as np
A = np.random.randint(0, 5, (5, 4))
A

array([[0, 3, 0, 0],
       [3, 2, 0, 1],
       [0, 3, 4, 0],
       [2, 3, 2, 4],
       [1, 4, 3, 2]])

In [2]:
x = np.random.randn(4, 1)
x

array([[ 0.072974  ],
       [-0.17768729],
       [ 0.23013913],
       [ 0.29585426]])

In [3]:
np.linalg.norm(x, 2)

np.float64(0.42117898672037785)

In [4]:
np.linalg.norm(A.dot(x), 2)

np.float64(1.5664931356188379)

In [5]:
np.linalg.svd(A)

SVDResult(U=array([[-2.30072954e-01,  9.56430753e-02, -7.42523863e-01,
         5.63570434e-01, -2.62612866e-01],
       [-2.85823644e-01, -5.35446394e-01, -4.67951582e-01,
        -6.42357627e-01, -2.98221725e-17],
       [-4.45908217e-01,  6.87511030e-01,  5.25901418e-02,
        -4.12984635e-01, -3.93919299e-01],
       [-5.69899832e-01, -4.58275261e-01,  4.76057903e-01,
         2.88780931e-01, -3.93919299e-01],
       [-5.84595009e-01,  1.46498909e-01,  1.68160681e-02,
         1.25754959e-01,  7.87838597e-01]]), S=array([9.31027653, 3.91937101, 2.47020043, 1.68979042]), Vh=array([[-0.27731353, -0.71401427, -0.5023715 , -0.40112804],
       [-0.60631943,  0.12495357,  0.57993753, -0.52956192],
       [-0.17606785, -0.61139425,  0.49102274,  0.59505785],
       [-0.72420582,  0.31743864, -0.41254335,  0.45229042]]))

## pytorch api

In [6]:
import numpy as np
import torch 
from torch import nn

In [7]:
m = nn.Linear(5, 4)

In [8]:
W = m.weight.clone()
W

tensor([[ 0.4028, -0.0587,  0.2538,  0.1070, -0.0106],
        [ 0.3959,  0.3375,  0.2065, -0.1554, -0.3067],
        [-0.0758,  0.4029,  0.0872,  0.3135, -0.1935],
        [ 0.1404,  0.0274,  0.4022, -0.3517,  0.4444]],
       grad_fn=<CloneBackward0>)

In [9]:
m_sn = nn.utils.parametrizations.spectral_norm(m)
m_sn.weight

tensor([[ 0.5013, -0.0731,  0.3160,  0.1332, -0.0132],
        [ 0.4928,  0.4200,  0.2570, -0.1935, -0.3818],
        [-0.0943,  0.5015,  0.1085,  0.3902, -0.2408],
        [ 0.1748,  0.0341,  0.5006, -0.4377,  0.5532]], grad_fn=<DivBackward0>)

In [10]:
U, s, V = np.linalg.svd(W.detach().numpy())
s

array([0.8037147 , 0.7559412 , 0.40333235, 0.3285751 ], dtype=float32)

In [11]:
W/s[0]

tensor([[ 0.5011, -0.0731,  0.3158,  0.1331, -0.0132],
        [ 0.4926,  0.4199,  0.2569, -0.1934, -0.3816],
        [-0.0943,  0.5013,  0.1085,  0.3901, -0.2407],
        [ 0.1747,  0.0341,  0.5004, -0.4376,  0.5530]], grad_fn=<DivBackward0>)